# Lab 3.3.3 — Compare models and dataset combinations

**Hands-on objective:** observe how three fixed model/dataset
combinations affect measured fit time and classification behavior.

Follow this notebook from top to bottom. Every learner question is
numbered and gives the expected response type. `learning_log.md` holds
only the matching checkpoint responses and later retrieval.


## Learning agreement and evidence boundary

- Compare exactly candidates A, B, and C defined below.
- Build a fresh preprocessing/model pipeline for every candidate.
- Fit candidate parameters on training rows only and compare only on
  validation rows.
- Time only `.fit(...)` using `time.perf_counter()`.
- Select by validation F1; lower measured fit time breaks only an exact
  F1 tie.
- Lock the winner, refit it once on train plus validation, and only then
  open and evaluate test rows once.

This is not hyperparameter tuning, fairness testing, drift testing, or
deployment assessment. The historical Portuguese 2008–2010 data and
retained demographic/financial fields are a bounded educational
baseline, not an endorsement of real-world profiling.


In [1]:
# Preflight — run without edits. It validates the full handoff but exposes
# only training and validation rows at this stage.
import os
import platform
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from artifact_contract import (
    CATEGORICAL_FEATURES,
    FULL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    load_development_artifacts,
    open_test_frame,
)

LAB_ROOT = Path.cwd()
development = load_development_artifacts(LAB_ROOT)
train = development.train
validation = development.validation

assert len(train) == 24_712 and len(validation) == 8_238
assert set(train["source_row_id"]).isdisjoint(set(validation["source_row_id"]))
assert not {"duration", "source_row_id", "subscribed"} & set(FULL_FEATURES)

runtime_context = {
    "platform": platform.platform(),
    "processor": platform.processor() or os.environ.get("PROCESSOR_IDENTIFIER", "not reported"),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
print("Runtime context (timings apply only to this run):")
for key, value in runtime_context.items():
    print(f"  {key}: {value}")
print(f"Verified artifact directory: {development.context.artifact_dir}")
print("Preflight passed. Training and validation are open; test is sealed.")


Runtime context (timings apply only to this run):
  platform: Windows-11-10.0.26100-SP0
  processor: Intel64 Family 6 Model 181 Stepping 0, GenuineIntel
  python: 3.14.4
  pandas: 3.0.0
  scikit-learn: 1.9.0
  joblib: 1.5.3
Verified artifact directory: C:\Users\TommasoBrindani\OneDrive - intec cooperativa sociale\Obsidian OneDrive\00 - inTEC\03 - AI Testing\HandsOn-Excercises\03_02_02_prepare_ml_data\artifacts
Preflight passed. Training and validation are open; test is sealed.


## Checkpoint 1 — Predict before fitting

Record short predictions in **learning log Checkpoint 1**.

**Q1.1 (expected: one comparison sentence):** How might logistic
regression on the core 15 features differ from logistic regression on
all 20 features?

**Q1.2 (expected: one comparison sentence):** With the full feature set
fixed, how might logistic regression and a random forest differ in
validation behavior?

**Q1.3 (expected: a candidate plus a reason):** Which candidate do you
predict will take longest to fit?

**Q1.4 (expected: two environment factors):** Why will measured seconds
be specific to this computer and run?


## Checkpoint 2 — Define the two dataset combinations

The **core** combination removes only the five economic-context
features from the full ordered list.

**Q2.1 (expected: one Python list-comprehension expression):** Replace
`None` below to derive the ordered core list from `FULL_FEATURES`. Do not
type a separate hand-maintained feature list.


In [2]:
ECONOMIC_CONTEXT_FEATURES = {
    "emp_var_rate",
    "cons_price_idx",
    "cons_conf_idx",
    "euribor3m",
    "nr_employed"}
CORE_FEATURES = [
      feature for feature in FULL_FEATURES
      if feature not in ECONOMIC_CONTEXT_FEATURES
] # TODO: ordered full list minus the five names above

assert CORE_FEATURES is not None, "Q2.1 is unfinished: derive CORE_FEATURES."
assert CORE_FEATURES == [
    feature for feature in FULL_FEATURES
    if feature not in ECONOMIC_CONTEXT_FEATURES
], "The core feature membership or order is incorrect."
assert len(CORE_FEATURES) == 15 and len(FULL_FEATURES) == 20
assert not {"duration", "source_row_id", "subscribed"} & set(CORE_FEATURES)
print("Core features (15):", CORE_FEATURES)
print("Full features (20):", FULL_FEATURES)


Core features (15): ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'previously_contacted']
Full features (20): ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'previously_contacted']


Three candidates are fixed in advance:

| Candidate | Model | Dataset combination |
| --- | --- | --- |
| A | Logistic regression, fixed upstream settings | Core 15 features |
| B | Logistic regression, fixed upstream settings | Full 20 features |
| C | 200-tree random forest, fixed settings | Full 20 features |

**Q2.2 (expected: two candidate pairs in learning log Checkpoint 2):**
Identify the pair that isolates the dataset combination and the pair
that isolates the model family.

**Q2.3 (expected: two prose sentences in learning log Checkpoint 2):**
Explain why preprocessing must be fresh for each candidate and which
rows may determine its imputation, encoding, scaling, and model state.


In [3]:
CANDIDATES = [
    {"candidate": "A", "model_kind": "logistic", "features": CORE_FEATURES},
    {"candidate": "B", "model_kind": "logistic", "features": FULL_FEATURES},
    {"candidate": "C", "model_kind": "random_forest", "features": FULL_FEATURES},
]

def build_candidate_pipeline(candidate):
    features = list(candidate["features"])
    numeric_features = [name for name in NUMERIC_FEATURES if name in features]
    categorical_features = [name for name in CATEGORICAL_FEATURES if name in features]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if candidate["model_kind"] == "logistic":
        numeric_steps.append(("scaler", StandardScaler()))
        classifier = LogisticRegression(
            solver="lbfgs",
            C=1.0,
            max_iter=2000,
            class_weight=None,
        )
    elif candidate["model_kind"] == "random_forest":
        classifier = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=1,
            class_weight=None,
        )
    else:
        raise ValueError(f"Unsupported fixed candidate: {candidate!r}")

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier),
    ])

assert [candidate["candidate"] for candidate in CANDIDATES] == ["A", "B", "C"]
print("Candidate definitions and fresh-pipeline factory are ready.")


Candidate definitions and fresh-pipeline factory are ready.


## Fit and evaluate the three development candidates

**Q2.4 (expected: four small Python expressions):** Complete the marked
timing, prediction, and result expressions. The timer must surround only
`.fit(...)`; validation prediction and metric calculations happen after
the elapsed fit time is captured.


In [4]:
development_results = []
development_pipelines = []

for candidate in CANDIDATES:
    features = list(candidate["features"])
    candidate_pipeline = build_candidate_pipeline(candidate)
    development_pipelines.append(candidate_pipeline)

    fit_started = time.perf_counter()  # TODO: time.perf_counter()
    candidate_pipeline.fit(train[features], train[TARGET_COLUMN])
    fit_seconds = time.perf_counter() - fit_started  # TODO: time.perf_counter() - fit_started

    validation_predictions = candidate_pipeline.predict(validation[features])  # TODO: predict validation features
    result = {
        "candidate": candidate["candidate"],
        "model": candidate["model_kind"],
        "feature_count": len(features),
        "accuracy": accuracy_score(
      validation[TARGET_COLUMN], validation_predictions),
     # TODO: accuracy_score(...)
        "precision": precision_score(
      validation[TARGET_COLUMN], validation_predictions, pos_label=1, zero_division=0),
    # TODO: precision_score(..., pos_label=1, zero_division=0)
        "recall": recall_score(
      validation[TARGET_COLUMN], validation_predictions, pos_label=1, zero_division=0),
       # TODO: recall_score(..., pos_label=1, zero_division=0)
        "f1": f1_score(
      validation[TARGET_COLUMN], validation_predictions, pos_label=1, zero_division=0),
           # TODO: f1_score(..., pos_label=1, zero_division=0)
        "fit_seconds": fit_seconds,
    }
    assert fit_started is not None and fit_seconds is not None and fit_seconds >= 0
    assert validation_predictions is not None
    assert all(result[name] is not None for name in ["accuracy", "precision", "recall", "f1"]), (
        f"Q2.4 is unfinished for candidate {candidate['candidate']}."
    )
    assert all(0.0 <= result[name] <= 1.0 for name in ["accuracy", "precision", "recall", "f1"])
    development_results.append(result)

assert len(development_results) == 3
assert len({id(pipeline) for pipeline in development_pipelines}) == 3
assert len({id(pipeline.named_steps["preprocessor"]) for pipeline in development_pipelines}) == 3

validation_results = pd.DataFrame(development_results).set_index("candidate")
display(validation_results[[
    "model", "feature_count", "accuracy", "precision", "recall", "f1", "fit_seconds"
]])


,model,feature_count,accuracy,precision,recall,f1,fit_seconds
candidate,,,,,,,
A,logistic,15,0.898155,0.661818,0.196121,0.302577,0.314049
B,logistic,20,0.899005,0.644578,0.230603,0.339683,0.495034
C,random_forest,20,0.891721,0.536437,0.285560,0.372714,8.457990


## Checkpoint 3 — Interpret validation evidence

Respond in **learning log Checkpoint 3** before selecting.

**Q3.1 (expected: one evidence sentence):** Which candidate has the
highest validation F1, and what does its precision/recall balance add to
the accuracy comparison?

**Q3.2 (expected: one pairwise interpretation):** Compare A with B to
describe the observed feature-set effect while the model stays fixed.

**Q3.3 (expected: one pairwise interpretation):** Compare B with C to
describe the observed model-family effect while the full dataset stays
fixed.

**Q3.4 (expected: a cautious timing statement plus runtime context):**
Report the measured fit times and explain why small differences should
not be generalized beyond this environment.


In [5]:
ranked_results = sorted(
    development_results,
    key=lambda result: (-result["f1"], result["fit_seconds"]),
)
winning_result = ranked_results[0]
winning_candidate = next(
    candidate for candidate in CANDIDATES
    if candidate["candidate"] == winning_result["candidate"]
)
exact_f1_tie = sum(
    result["f1"] == winning_result["f1"] for result in development_results
) > 1

print("Selection rule: highest validation F1; lower fit time breaks an exact F1 tie.")
print("Validation winner:", winning_result["candidate"])
print(f"Winning validation F1: {winning_result['f1']:.6f}")
print("Fit-time tie-break used:", exact_f1_tie)


Selection rule: highest validation F1; lower fit time breaks an exact F1 tie.
Validation winner: C
Winning validation F1: 0.372714
Fit-time tie-break used: False


### Immediate selection check

**Q3.5 (expected: one evidence sentence in learning log Checkpoint 3):** State whether an exact validation-F1 tie occurred and therefore whether measured fit time decided the winner.

## Checkpoint 4 — Lock the winner before opening test

Respond in **learning log Checkpoint 4** now.

**Q4.1 (expected: candidate ID and rule):** Record the selected candidate
and state how the validation rule selected it.

**Q4.2 (expected: numerical prediction):** Predict final test F1 and the
difference `test F1 − selected validation F1`.

**Q4.3 (expected: two prose sentences):** State what a lower test result
could mean and why it would not authorize choosing a different candidate
from this same test evidence.

**Q4.4 (expected: one evidence-role sentence):** Explain why the test set still has an independent role immediately before it is opened.

Set the confirmation to `True` only after writing. The next code first
locks the selection and refits a **fresh** winning configuration on train
plus validation. Test rows remain unopened during that refit.


In [6]:
I_RECORDED_AND_ACCEPTED_THE_VALIDATION_WINNER = True  # TODO

assert I_RECORDED_AND_ACCEPTED_THE_VALIDATION_WINNER, (
    "Checkpoint 4 is unfinished. Record the winner and test prediction before continuing."
)
locked_candidate_id = winning_candidate["candidate"]
locked_validation_f1 = winning_result["f1"]
locked_candidate = {
    "candidate": winning_candidate["candidate"],
    "model_kind": winning_candidate["model_kind"],
    "features": list(winning_candidate["features"]),
}
selection_locked = True

final_pipeline = build_candidate_pipeline(locked_candidate)
assert all(final_pipeline is not pipeline for pipeline in development_pipelines)
train_validation = pd.concat([train, validation], ignore_index=True)
assert len(train_validation) == 32_950
assert train_validation["source_row_id"].is_unique

final_features = locked_candidate["features"]
assert not {"duration", "source_row_id", "subscribed"} & set(final_features)
final_pipeline.fit(
    train_validation[final_features],
    train_validation[TARGET_COLUMN],
)
print(f"Locked {locked_candidate_id} and refitted one fresh pipeline on train + validation.")
print("Test rows are still sealed.")


Locked C and refitted one fresh pipeline on train + validation.
Test rows are still sealed.


## Open the test vault once

The selection and fresh refit are complete. Run the next cell once. It
verifies that the upstream artifacts have not changed, opens the fixed
test membership, obtains one prediction set, and calculates the four
final metrics. No code after it changes the selection.


In [7]:
test = open_test_frame(
    development.context,
    selection_locked=selection_locked,
)
assert len(test) == 8_238
assert set(test["source_row_id"]).isdisjoint(set(train_validation["source_row_id"]))

test_predictions = final_pipeline.predict(test[final_features])
final_test_metrics = {
    "accuracy": accuracy_score(test[TARGET_COLUMN], test_predictions),
    "precision": precision_score(
        test[TARGET_COLUMN], test_predictions, pos_label=1, zero_division=0
    ),
    "recall": recall_score(
        test[TARGET_COLUMN], test_predictions, pos_label=1, zero_division=0
    ),
    "f1": f1_score(
        test[TARGET_COLUMN], test_predictions, pos_label=1, zero_division=0
    ),
}
assert all(0.0 <= value <= 1.0 for value in final_test_metrics.values())

print(f"Final locked candidate: {locked_candidate_id}")
for metric_name, value in final_test_metrics.items():
    print(f"  test {metric_name:9s}: {value:.6f}")
print(f"  test F1 - selected validation F1: {final_test_metrics['f1'] - locked_validation_f1:+.6f}")


Final locked candidate: C
  test accuracy : 0.896213
  test precision: 0.576842
  test recall   : 0.295259
  test f1       : 0.390592
  test F1 - selected validation F1: +0.017877


## Checkpoint 5 — Interpret once, then stop

Respond in **learning log Checkpoint 5**.

**Q5.1 (expected: one concise result sentence):** Report the locked
candidate's four test metrics and the test-minus-validation F1
difference.

**Q5.2 (expected: one workflow explanation):** Why was a fresh pipeline
refitted on train plus validation only after selection?

**Q5.3 (expected: one boundary statement):** Explain why test results
cannot now change the candidate, feature set, seed, threshold, or model.

Change the confirmation below only after writing.


In [9]:
I_COMPLETED_THE_FINAL_INTERPRETATION = True  # TODO

assert I_COMPLETED_THE_FINAL_INTERPRETATION, (
    "Checkpoint 5 is unfinished. Interpret the single final test before completing the lab."
)
print("HO-3.3.3 complete: one validation-locked configuration received one final test.")


HO-3.3.3 complete: one validation-locked configuration received one final test.


## Completion check

You are finished when you can explain what A-vs-B and B-vs-C isolate,
why timing is environment-specific, how the F1/timing selection rule was
applied, and why no choice may now change. Close the notebook and answer
the five retrieval questions at the end of `learning_log.md` without
reopening code first.
